# 19a — Introduction to Convolutional Neural Networks

Corresponds to: Note 19 — Introduction to Convolutional Neural Networks

**Libraries introduced in this notebook:** `numpy` — already introduced in Note 03. No new libraries.

**What this notebook demonstrates:**
1. The parameter explosion of fully connected layers on images — concrete numbers
2. Local receptive fields — weight count per neuron vs FC
3. The convolution operation — sliding filter, element-wise multiply-and-sum
4. Parameter sharing — one filter applied at every position
5. Translation equivariance — same filter detects the same feature at different positions

In [ ]:
import numpy as np

print('Note 19 — Introduction to CNNs')
print('numpy:', np.__version__)

## Part 1: The FC Parameter Explosion

Note 19 Part 1 shows that a single FC layer on a 224×224 RGB image requires tens of millions of weights. We compute the exact numbers.

In [ ]:
# Image dimensions
H, W, C = 224, 224, 3
n_pixels = H * W * C

# FC layer sizes
fc_512  = n_pixels * 512
fc_1024 = n_pixels * 1024

print(f'Image: {H}x{W}x{C} = {n_pixels:,} input values')
print()
print(f'FC layer with 512 neurons:   {fc_512:,} weights')
print(f'FC layer with 1024 neurons:  {fc_1024:,} weights')
print()

# CNN comparison: 3x3 filter, 32 output channels
cnn_params = (3 * 3 * C + 1) * 32
print(f'Conv layer (3x3, C_in=3, K=32): {cnn_params} weights')
print(f'Ratio FC/CNN: {fc_512 // cnn_params:,}x more parameters in FC')

## Part 2: Local Receptive Fields

A CNN neuron connects to a small k×k patch instead of all N pixels. We compare the weight count per neuron.

In [ ]:
print('Weights per neuron:')
print(f'  FC neuron (224x224 image):  {n_pixels:,}')
for k in [3, 5, 7]:
    print(f'  CNN neuron ({k}x{k} receptive field, single channel): {k*k}')
print()

# Number of positions a 3x3 filter can be placed on a 224x224 image (no padding)
positions = (224 - 3 + 1) ** 2
print(f'Positions for a 3x3 filter on 224x224 (no padding): {positions:,}')
print(f'Without parameter sharing: {positions * 9:,} weights for one filter')
print(f'With parameter sharing:    9 weights for one filter')
print(f'Reduction factor: {positions:,}x')

## Part 3: The Convolution Operation

A filter slides over the input, computing an element-wise multiply-and-sum at each position. We implement this on the 5×5 example from Note 19 Part 3.

In [ ]:
# 5x5 input, 3x3 filter — Note 19 Part 3 example
inp = np.array([
    [1, 2, 3, 0, 1],
    [0, 1, 2, 3, 0],
    [1, 0, 1, 2, 1],
    [2, 1, 0, 1, 2],
    [0, 1, 2, 1, 0],
])

filt = np.array([
    [1, 1, 0],
    [0, 1, 1],
    [1, 0, 1],
])

F = 3
H_out = inp.shape[0] - F + 1
W_out = inp.shape[1] - F + 1
print(f'Input: {inp.shape},  Filter: {filt.shape}')
print(f'Output size: ({inp.shape[0]} - {F} + 1) = {H_out}x{W_out}')
print()

feature_map = np.zeros((H_out, W_out), dtype=int)
for i in range(H_out):
    for j in range(W_out):
        patch = inp[i:i+F, j:j+F]
        feature_map[i, j] = (patch * filt).sum()

print('Feature map (3x3):')
print(feature_map)
print()

# Show step-by-step for position (0,0)
patch = inp[0:3, 0:3]
products = patch * filt
print('Position (0,0) step-by-step:')
print(f'  Patch:    {patch[0]}   Filter: {filt[0]}   Products: {products[0]}')
print(f'            {patch[1]}           {filt[1]}             {products[1]}')
print(f'            {patch[2]}           {filt[2]}             {products[2]}')
print(f'  Sum = {products.sum()}')

## Part 4: Parameter Sharing

The same filter weights are used at every position. We verify this by applying the filter at all 9 positions and confirming the same 9 weights produce the full feature map.

In [ ]:
print('Filter weights (shared across all positions):')
print(filt)
print(f'Total weights for this filter: {filt.size}  (regardless of input size)')
print()
print('Output at each position (same filter used everywhere):')
print(f'{"Position":<12} {"Patch sum":<12} {"Output"}')
print('-' * 36)
for i in range(H_out):
    for j in range(W_out):
        patch = inp[i:i+F, j:j+F]
        val = (patch * filt).sum()
        print(f'({i},{j}){"":<8} {val:<12} {val}')

## Part 5: Translation Equivariance

A vertical edge filter responds strongly wherever a vertical edge appears, regardless of position. We demonstrate this with a simple synthetic example.

In [ ]:
# Vertical edge filter: responds to left-bright, right-dark transitions
edge_filter = np.array([
    [ 1, -1],
    [ 1, -1],
    [ 1, -1],
], dtype=float)

# Image A: vertical edge on the left
img_a = np.array([
    [1, 1, 0, 0, 0, 0],
    [1, 1, 0, 0, 0, 0],
    [1, 1, 0, 0, 0, 0],
    [1, 1, 0, 0, 0, 0],
], dtype=float)

# Image B: same edge shifted to the right
img_b = np.array([
    [0, 0, 0, 1, 1, 0],
    [0, 0, 0, 1, 1, 0],
    [0, 0, 0, 1, 1, 0],
    [0, 0, 0, 1, 1, 0],
], dtype=float)

def convolve2d(img, filt):
    fh, fw = filt.shape
    oh = img.shape[0] - fh + 1
    ow = img.shape[1] - fw + 1
    out = np.zeros((oh, ow))
    for i in range(oh):
        for j in range(ow):
            out[i, j] = (img[i:i+fh, j:j+fw] * filt).sum()
    return out

resp_a = convolve2d(img_a, edge_filter)
resp_b = convolve2d(img_b, edge_filter)

print('Vertical edge filter response:')
print()
print('Image A (edge on left):')
print(img_a.astype(int))
print('Response:')
print(resp_a.astype(int))
print(f'Max response at column: {resp_a.max(axis=0).argmax()}')
print()
print('Image B (edge shifted right):')
print(img_b.astype(int))
print('Response:')
print(resp_b.astype(int))
print(f'Max response at column: {resp_b.max(axis=0).argmax()}')
print()
print('Same filter, same response magnitude — just at a different position.')
print(f'Max response A: {resp_a.max():.1f}  Max response B: {resp_b.max():.1f}')

## Summary

This notebook demonstrated:

1. **FC parameter explosion** — a single FC layer on a 224×224 RGB image requires 77M+ weights for 512 neurons; a conv layer with the same output depth needs only 896 weights.
2. **Local receptive fields** — a CNN neuron with a 3×3 receptive field uses 9 weights instead of 150,528; parameter sharing reduces this further to 9 weights total per filter.
3. **Convolution** — the filter slides over the input, computing element-wise multiply-and-sum at each position to produce a feature map.
4. **Parameter sharing** — the same 9 filter weights are reused at all positions; the feature map is the result of applying one set of weights everywhere.
5. **Translation equivariance** — a vertical edge filter produces the same response magnitude wherever the edge appears; the response shifts with the feature, it does not disappear.